
# Financial model  




In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import brentq
from pathlib import Path
import pandas as pd
from typing import Iterable

from config import INPUT_DIR, FINANCIAL, HYDRO_FIT_EUR_MWH

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## Helper functions

In [ ]:
def compute_average_yearly_da_curve(da_prices: Iterable[float], hours_per_year: int = 8760) -> np.ndarray:
    arr = np.asarray(list(da_prices), dtype=float).reshape(-1)
    if arr.size < hours_per_year:
        raise ValueError("da_prices must contain at least one full year of hourly prices.")

    n_years = arr.size // hours_per_year
    trimmed = arr[: n_years * hours_per_year]
    matrix = trimmed.reshape(n_years, hours_per_year)
    return matrix.mean(axis=0)


def compute_market_based_ppa_from_da_prices(
    da_prices: Iterable[float],
    hours_per_year: int = 8760,
    lower_fraction: float = 0.50,
) -> float:
    """
    Scenario 2.2 PPA definition:
    average of the lower 50% prices from the average yearly DA curve.
    """
    if not (0.0 < lower_fraction <= 1.0):
        raise ValueError("lower_fraction must be in (0, 1].")

    avg_curve = compute_average_yearly_da_curve(da_prices, hours_per_year=hours_per_year)
    sorted_curve = np.sort(avg_curve)
    n_keep = max(1, int(np.floor(lower_fraction * sorted_curve.size)))
    return float(sorted_curve[:n_keep].mean())

def compute_npv(cash_flows: np.ndarray, discount_rate: float) -> float:
    cash_flows = np.asarray(cash_flows, dtype=float)
    t = np.arange(len(cash_flows), dtype=float)
    return float(np.sum(cash_flows / (1.0 + discount_rate) ** t))


def compute_project_irr(cash_flows: np.ndarray) -> float:
    """Annual IRR: solves NPV(rate)=0 for equally spaced annual cash flows."""
    cash_flows = np.asarray(cash_flows, dtype=float)

    if not (np.any(cash_flows < 0) and np.any(cash_flows > 0)):
        return np.nan

    def npv_at(rate: float) -> float:
        t = np.arange(len(cash_flows), dtype=float)
        return float(np.sum(cash_flows / (1.0 + rate) ** t))

    # For the ranking we only care about positive IRRs.
    if npv_at(0.0) <= 0.0:
        return np.nan

    try:
        # 10.0 = 1000% upper bound, high enough for project screening.
        return float(brentq(npv_at, 0.0, 10.0, xtol=1e-9, maxiter=500))
    except ValueError:
        return np.nan


## Compute the market-derived PPA from DA prices

In [ ]:

da_path = INPUT_DIR / "Prices Central" / "DA.csv"
# Lower fraction used to compute the market-derived PPA
lower_fraction = 0.50

da_df = pd.read_csv(da_path, sep=";", decimal=",")
da_df = da_df.dropna(axis=1, how="all")
da_numeric = da_df.apply(pd.to_numeric, errors="coerce")

# Average yearly curve across all DA years
avg_yearly_curve = da_numeric.mean(axis=1).to_numpy(dtype=float)

ppa_market_driven_list = []
percentile = [0.20, 0.35, 0.50, 0.65, 0.80]
for p in percentile:
    ppa = compute_market_based_ppa_from_da_prices(
        da_prices=avg_yearly_curve,
        lower_fraction=p,
    )
    ppa_market_driven_list.append(ppa)
    print(f"Market-derived PPA at {int(p*100)}th percentile: {ppa:.2f} €/MWh")    
print(ppa_market_driven_list)

In [ ]:
sensibility_ppa_market_driven_list = []
percentile = [0.10, 0.20,0.30,0.40, 0.50, 0.60, 0.70,0.80, 0.90]
for p in percentile:
    ppa = compute_market_based_ppa_from_da_prices(
        da_prices=avg_yearly_curve,
        lower_fraction=p,
    )
    sensibility_ppa_market_driven_list.append(ppa)
    print(f"Market-derived PPA at {int(p*100)}th percentile: {ppa:.2f} €/MWh")    
print(sensibility_ppa_market_driven_list)

## Build base-case cash flows


In [ ]:
# Base case years: 2025–2044
base_years = list(range(2025, 2045))

# --- Hydro production directly from input hydro CSV ---
hydro_input = pd.read_csv(
    "Inputs/Hydro_canedo.csv",   # change path if needed
    sep=";",
    decimal=","
)

hydro_year_cols = [str(y) for y in base_years]

# Convert hydro power profile [MW] into annual energy [MWh]
hydro_prod_all = hydro_input[hydro_year_cols].sum(axis=0).to_numpy(dtype=float) 

# --- DA prices only for 2035–2044 ---
da_year_cols_DA = [str(y) for y in range(2035, 2045)]
da_prices_DA = da_numeric[da_year_cols_DA].mean(axis=0).to_numpy(dtype=float)

# --- Inflation-adjusted hydro OPEX ---
inflation_factors = (1.0 + FINANCIAL.inflation_rate) ** np.arange(len(base_years))
hydro_opex_all = FINANCIAL.opex_hydro_eur_per_year * inflation_factors

# --- Base case revenues ---
base_revenue_all = np.zeros(len(base_years), dtype=float)

# 2025–2034: FiT 
fit_mask = np.array([(2025 <= y <= 2034) for y in base_years])
base_revenue_all[fit_mask] = hydro_prod_all[fit_mask] * HYDRO_FIT_EUR_MWH

# 2035–2044: DA market
da_mask = np.array([(2035 <= y <= 2044) for y in base_years])
base_revenue_all[da_mask] = hydro_prod_all[da_mask] * da_prices_DA

# --- Final base case cash flows ---
ebitda_base_array = base_revenue_all - hydro_opex_all #ebitda = ebit, no depreciation considered for the hydro
tax = np.maximum(0.0, ebitda_base_array * FINANCIAL.corporate_tax_rate)
cf_base_array = ebitda_base_array - tax

base_cashflows = pd.DataFrame({
    "calendar_year": base_years,
    "hydro_production_mwh": hydro_prod_all,
    "base_revenue_eur": base_revenue_all,
    "hydro_opex_eur": hydro_opex_all,
    "project_cash_flow_eur": cf_base_array,
})

print("HYDRO_FIT_EUR_MWH:", HYDRO_FIT_EUR_MWH)
display(base_cashflows)

## IRR Computation

In [ ]:
RESULTS_DIR = Path("results") / "Scenario 2 results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


_all_B = pd.read_csv("results/dispatch_scenario_B.csv", sep=";", decimal=".", encoding="utf-8-sig")
_all_D = pd.read_csv("results/dispatch_scenario_D.csv", sep=";", decimal=".", encoding="utf-8-sig")


print(_all_B.columns.tolist())
print(_all_D.columns.tolist())

In [ ]:
_PROJECT_START = 2025
_PROJECT_END = 2044
_FIRST_OPERATION_YEAR = 2027
_DISCOUNT_RATE = 

PPA_price = ppa_market_driven_list[2]  # using the 50th percentile market-derived PPA as an example

project_years = list(range(_PROJECT_START, _PROJECT_END + 1))
operation_years = list(range(_FIRST_OPERATION_YEAR, _PROJECT_END + 1))
# ────────────────────────────────────────────────────────────────────────────


# --- CSV reader

def read_dispatch_csv(pa_th: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", decimal=".", encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    return df


# Keep only successful dispatch runs, if the status column exists
if "status" in _all_B.columns:
    _all_B = _all_B[_all_B["status"].eq("ok")].copy()

if "status" in _all_D.columns:
    _all_D = _all_D[_all_D["status"].eq("ok")].copy()


# --- Configuration keys ---
_key_cols = ["pv_mw", "bess_power_mw", "bess_duration_h", "contract_share"]

# Only evaluate configurations that exist in both B and D
_configs = (
    _all_B[_key_cols].drop_duplicates()
    .merge(_all_D[_key_cols].drop_duplicates(), on=_key_cols, how="inner")
    .sort_values(_key_cols)
    .reset_index(drop=True)
)


# --- Helper functions ---
def get_bess_capex_eur(bess_power_mw: float, bess_duration_h: float) -> float:
    """BESS CAPEX is defined in the config in €/MW of BESS power."""
    if np.isclose(bess_power_mw, 0.0):
        return 0.0

    if np.isclose(bess_duration_h, 2.0):
        return FINANCIAL.capex_2hbess_eur_per_mw * bess_power_mw

    if np.isclose(bess_duration_h, 4.0):
        return FINANCIAL.capex_4hbess_eur_per_mw * bess_power_mw

    raise ValueError(f"Unsupported BESS duration: {bess_duration_h}")


def get_bess_opex_eur_per_year(bess_power_mw, bess_duration_h):
    if bess_power_mw == 0:
        return 0.0

    if np.isclose(bess_duration_h, 2.0):
        return FINANCIAL.opex_2hbess_eur_per_mw_year * bess_power_mw

    if np.isclose(bess_duration_h, 4.0):
        return FINANCIAL.opex_4hbess_eur_per_mw_year * bess_power_mw

    raise ValueError(f"Unsupported BESS duration: {bess_duration_h}")


def build_depreciation_series(capex_eur: float, depreciation_years: int, n_operation_years: int) -> np.ndarray:
    """Straight-line depreciation starting in the first operating year."""
    depreciation = np.zeros(n_operation_years, dtype=float)
    if capex_eur <= 0 or depreciation_years <= 0:
        return depreciation

    n = min(depreciation_years, n_operation_years)
    depreciation[:n] = capex_eur / depreciation_years
    return depreciation


def compute_after_tax_operating_cf(revenue_eur: np.ndarray, opex_eur: np.ndarray, depreciation_eur: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Project-finance operating cash flow:
        EBITDA = revenue - OPEX
        EBIT = EBITDA - depreciation
        tax = max(0, EBIT * tax_rate)
        operating CF = EBITDA - tax

    Depreciation is not a cash outflow; it only creates a tax shield.
    """
    ebitda = revenue_eur - opex_eur
    ebit = ebitda - depreciation_eur
    tax = np.maximum(0.0, ebit * FINANCIAL.corporate_tax_rate)
    operating_cf = ebitda - tax
    return operating_cf, ebitda, ebit, tax


_sweep_rows = []

for _, _cfg in _configs.iterrows():

    _pv = float(_cfg["pv_mw"])
    _bp = float(_cfg["bess_power_mw"])
    _bd = float(_cfg["bess_duration_h"])
    _cs = float(_cfg["contract_share"])

    # --- Select this configuration in B and D ---
    _mB = (
        np.isclose(_all_B["pv_mw"], _pv)
        & np.isclose(_all_B["bess_power_mw"], _bp)
        & np.isclose(_all_B["bess_duration_h"], _bd)
        & np.isclose(_all_B["contract_share"], _cs)
    )

    _mD = (
        np.isclose(_all_D["pv_mw"], _pv)
        & np.isclose(_all_D["bess_power_mw"], _bp)
        & np.isclose(_all_D["bess_duration_h"], _bd)
        & np.isclose(_all_D["contract_share"], _cs)
    )

    _sub_B = _all_B[_mB].sort_values("year").copy()
    _sub_D = _all_D[_mD].sort_values("year").copy()

    # Scenario B must cover 2027–2034; Scenario D must cover 2035–2044
    if len(_sub_B) != 8 or len(_sub_D) != 10:
        continue

    if set(_sub_B["year"].astype(int)) != set(range(2027, 2035)):
        continue

    if set(_sub_D["year"].astype(int)) != set(range(2035, 2045)):
        continue

    try:
        # ------------------------------------------------------------------
        # 1. Incremental CAPEX during construction years: 2025–2026
        # ------------------------------------------------------------------
        pv_capex_eur = (
            FINANCIAL.capex_pv_eur_per_mw
            + FINANCIAL.capex_epc_contingency
        ) * _pv

        bess_capex_eur = get_bess_capex_eur(_bp, _bd)
        total_capex_eur = FINANCIAL.line_capex_fixed + pv_capex_eur + bess_capex_eur
        
       
        cf_construction = np.array([
            -total_capex_eur / 2.0,
            -total_capex_eur / 2.0,
        ], dtype=float)

        # ------------------------------------------------------------------
        # 2. Depreciation series for the 18 operating years, 2027–2044
        # ------------------------------------------------------------------
        n_operation_years = len(operation_years)

        pv_depreciation = build_depreciation_series(
            capex_eur=pv_capex_eur,
            depreciation_years=FINANCIAL.depreciation_years_pv,
            n_operation_years=n_operation_years,
        )

        bess_depreciation = build_depreciation_series(
            capex_eur=bess_capex_eur,
            depreciation_years=FINANCIAL.depreciation_years_bess,
            n_operation_years=n_operation_years,
        )

        total_depreciation = pv_depreciation + bess_depreciation

        depreciation_B  = total_depreciation[:8]
        depreciation_D = total_depreciation[8:]

        # ------------------------------------------------------------------
        # 3. Scenario B incremental operating cash flows: 2027–2034
        # ------------------------------------------------------------------
        years_B = _sub_B["year"].to_numpy(dtype=int)

        da_prices_B = (
            da_numeric[[str(y) for y in years_B]]
            .mean(axis=0)
            .to_numpy(dtype=float)
        )

        inflation_B = (1.0 + FINANCIAL.inflation_rate) ** (years_B - _PROJECT_START)

        fixed_opex_B = (
            FINANCIAL.opex_pv_eur_per_mw_year * _pv
            + get_bess_opex_eur_per_year(_bp, _bd)
        ) * inflation_B

        # Incremental variable OPEX: avoid charging the existing hydro output.
        incremental_output_B = np.maximum(
            _sub_B["total_output_mwh"].to_numpy(dtype=float)
            - _sub_B["hydro_annual_mwh"].to_numpy(dtype=float),
            0.0,
        )

        variable_opex_B = (
            FINANCIAL.opex_balancing_eur_per_mwh
            * incremental_output_B
            * inflation_B
        )

        grid_tariffs_bess_charge_B = (
                (_sub_B["bess_charge_mwh"].to_numpy(dtype=float) #total charge
                 - ((_sub_B["pv_annual_mwh"].to_numpy(dtype=float) - _sub_B["pv_curtailed_mwh"].to_numpy(dtype=float)) #pv charge bess
                    + _sub_B["hydro_to_bess_mwh"].to_numpy(dtype=float) # hydro charge bess
                    ) #bess charge from grid = total charge - charge from pv - charge from hydro
                 )*FINANCIAL.grid_tariff_bess_charge_eur_per_mwh
        )

        revenue_B = (
            _sub_B["merchant_revenue_eur"].to_numpy(dtype=float)
            + _sub_B["ppa_delivered_mwh"].to_numpy(dtype=float) * PPA_price
            - _sub_B["ppa_shortfall_mwh"].to_numpy(dtype=float) * da_prices_B
        )

        operating_cf_B, ebitda_B, ebit_B, tax_B = compute_after_tax_operating_cf(
            revenue_eur=revenue_B,
            opex_eur=fixed_opex_B + variable_opex_B + grid_tariffs_bess_charge_B,
            depreciation_eur=depreciation_B,
        )

        # ------------------------------------------------------------------
        # 4. Scenario D incremental operating cash flows: 2035–2044
        # ------------------------------------------------------------------
        years_D = _sub_D["year"].to_numpy(dtype=int)

        da_prices_D = (
            da_numeric[[str(y) for y in years_D]]
            .mean(axis=0)
            .to_numpy(dtype=float)
        )

        inflation_D = (1.0 + FINANCIAL.inflation_rate) ** (years_D - _PROJECT_START)

        fixed_opex_D = (
            FINANCIAL.opex_pv_eur_per_mw_year * _pv
            + get_bess_opex_eur_per_year(_bp, _bd)
        ) * inflation_D

        incremental_output_D = np.maximum(
            _sub_D["total_output_mwh"].to_numpy(dtype=float)
            - _sub_D["hydro_annual_mwh"].to_numpy(dtype=float),
            0.0,
        )

        variable_opex_D = (
            FINANCIAL.opex_balancing_eur_per_mwh
            * incremental_output_D
            * inflation_D
        )

        grid_tariffs_bess_charge_D = (
            (_sub_D["bess_charge_mwh"].to_numpy(dtype=float) #total charge
                - ((_sub_D["pv_annual_mwh"].to_numpy(dtype=float) - _sub_D["pv_curtailed_mwh"].to_numpy(dtype=float)) #pv charge bess
                + _sub_D["hydro_to_bess_mwh"].to_numpy(dtype=float) # hydro charge bess
                    ) #bess charge from grid = total charge - charge from pv - charge from hydro
                 )*FINANCIAL.grid_tariff_bess_charge_eur_per_mwh
        )

        # Scenario D merchant_revenue_eur includes the existing hydro as merchant.
        # Therefore, subtract the hydro-only DA counterfactual to isolate the
        # incremental hybridisation value.
        hydro_idx_D = [base_years.index(y) for y in years_D]
        hydro_only_da_revenue_D = hydro_prod_all[hydro_idx_D] * da_prices_D

        revenue_D = (
            _sub_D["merchant_revenue_eur"].to_numpy(dtype=float)
            + _sub_D["ppa_delivered_mwh"].to_numpy(dtype=float) * PPA_price
            - _sub_D["ppa_shortfall_mwh"].to_numpy(dtype=float) * da_prices_D
            - hydro_only_da_revenue_D
        )

        operating_cf_D, ebitda_D, ebit_D, tax_D = compute_after_tax_operating_cf(
            revenue_eur=revenue_D,
            opex_eur=fixed_opex_D + variable_opex_D + grid_tariffs_bess_charge_D,
            depreciation_eur=depreciation_D,
        )
       
        # ------------------------------------------------------------------
        # 5. Full hybridisation-only project cash flows: 2025–2044
        # ------------------------------------------------------------------
        project_cf = np.concatenate([
            cf_construction,
            operating_cf_B,
            operating_cf_D, 
        ])

        if len(project_cf) != len(project_years):
            raise ValueError(
                f"Project cash-flow length mismatch: {len(project_cf)} values "
                f"for {len(project_years)} project years."
            )

        hybridisation_IRR = compute_project_irr(project_cf)
        project_NPV = compute_npv(project_cf, _DISCOUNT_RATE)

        avg_revenue = float(np.mean(np.concatenate([revenue_B, revenue_D])))
        avg_ebitda = float(np.mean(np.concatenate([ebitda_B, ebitda_D])))
        avg_tax = float(np.mean(np.concatenate([tax_B, tax_D])))

    except Exception as _e:
        print(f"Error for pv={_pv}, bess={_bp}, duration={_bd}, c={_cs}: {_e}")

        hybridisation_IRR = np.nan
        project_NPV = np.nan
        total_capex_eur = np.nan
        avg_revenue = np.nan
        avg_ebitda = np.nan
        avg_tax = np.nan

    _sweep_rows.append({
        "pv_mw": _pv,
        "bess_mw": _bp,
        "bess_h": _bd,
        "contract_share": _cs,
        "capex_kEUR": None if np.isnan(total_capex_eur) else round(total_capex_eur / 1e3),
        "hybridisation_IRR": hybridisation_IRR,
        f"NPV_@{int(_DISCOUNT_RATE * 100)}pct_kEUR":
            None if np.isnan(project_NPV) else round(project_NPV / 1e3),
        "avg_revenue_kEUR_yr": None if np.isnan(avg_revenue) else round(avg_revenue / 1e3),
        "avg_EBITDA_kEUR_yr": None if np.isnan(avg_ebitda) else round(avg_ebitda / 1e3),
        "avg_tax_kEUR_yr": None if np.isnan(avg_tax) else round(avg_tax / 1e3),
    })


# --------------------------------------------------------------------------
# Ranking table
# --------------------------------------------------------------------------
df_sweep = (
    pd.DataFrame(_sweep_rows)
    .sort_values("hybridisation_IRR", ascending=False, na_position="last")
    .reset_index(drop=True)
)

df_sweep.index += 1


IRR_analysis_scenario2 = df_sweep.copy()
# Save export files
IRR_analysis_scenario2.to_csv(RESULTS_DIR /
    f"IRR_analysis_scenario2.csv",
    index=True,
    encoding="utf-8-sig"
)


# Table with Top20 
ranking_top = 20
df_top_x_export = (
    df_sweep
    .head(ranking_top)
    .copy()
)
display(df_top_x_export)



## PPA calculation for Target IRR

In [ ]:
target_irr_range = [0.16, 0.17, 0.18, 0.19, 0.2]

In [ ]:
def compute_required_ppa_for_target_irr(
    target_irr: float = 0.15,
    ppa_min: float = 0.0,
    ppa_max: float = 300.0,
    tol: float = 1e-5,
    max_iter: int = 100,
) -> pd.DataFrame:
    """
    Computes the required PPA price €/MWh for each configuration to reach target IRR.

    It uses the existing dispatch results from _all_B and _all_D.
    Dispatch is not re-optimized. Only the PPA price is changed in the financial model.
    """

    required_rows = []

    def build_project_cf_for_config(_cfg, ppa_price_eur_mwh):
        _pv = float(_cfg["pv_mw"])
        _bp = float(_cfg["bess_power_mw"])
        _bd = float(_cfg["bess_duration_h"])
        _cs = float(_cfg["contract_share"])

        _mB = (
            np.isclose(_all_B["pv_mw"], _pv)
            & np.isclose(_all_B["bess_power_mw"], _bp)
            & np.isclose(_all_B["bess_duration_h"], _bd)
            & np.isclose(_all_B["contract_share"], _cs)
        )

        _mD = (
            np.isclose(_all_D["pv_mw"], _pv)
            & np.isclose(_all_D["bess_power_mw"], _bp)
            & np.isclose(_all_D["bess_duration_h"], _bd)
            & np.isclose(_all_D["contract_share"], _cs)
        )

        _sub_B = _all_B[_mB].sort_values("year").copy()
        _sub_D = _all_D[_mD].sort_values("year").copy()

        if len(_sub_B) != 8 or len(_sub_D) != 10:
            raise ValueError("Incomplete scenario B or D years.")

        if set(_sub_B["year"].astype(int)) != set(range(2027, 2035)):
            raise ValueError("Scenario B does not cover 2027–2034.")

        if set(_sub_D["year"].astype(int)) != set(range(2035, 2045)):
            raise ValueError("Scenario D does not cover 2035–2044.")

        # CAPEX
        pv_capex_eur = (
            FINANCIAL.capex_pv_eur_per_mw
            + FINANCIAL.capex_epc_contingency
        ) * _pv

        bess_capex_eur = get_bess_capex_eur(_bp, _bd)

        total_capex_eur = (
            FINANCIAL.line_capex_fixed
            + pv_capex_eur
            + bess_capex_eur
        )
       
        cf_construction = np.array([
            -total_capex_eur / 2.0,
            -total_capex_eur / 2.0,
        ], dtype=float)

        cf_construction = np.array([
            -total_capex_eur / 2.0,
            -total_capex_eur / 2.0,
        ], dtype=float)

        # Depreciation
        n_operation_years = len(operation_years)

        pv_depreciation = build_depreciation_series(
            capex_eur=pv_capex_eur,
            depreciation_years=FINANCIAL.depreciation_years_pv,
            n_operation_years=n_operation_years,
        )

        bess_depreciation = build_depreciation_series(
            capex_eur=bess_capex_eur,
            depreciation_years=FINANCIAL.depreciation_years_bess,
            n_operation_years=n_operation_years,
        )

        total_depreciation = pv_depreciation + bess_depreciation

        depreciation_B = total_depreciation[:8]
        depreciation_D = total_depreciation[8:]

        # Scenario B
        years_B = _sub_B["year"].to_numpy(dtype=int)

        da_prices_B = (
            da_numeric[[str(y) for y in years_B]]
            .mean(axis=0)
            .to_numpy(dtype=float)
        )

        inflation_B = (1.0 + FINANCIAL.inflation_rate) ** (years_B - _PROJECT_START)

        fixed_opex_B = (
            FINANCIAL.opex_pv_eur_per_mw_year * _pv
            + get_bess_opex_eur_per_year(_bp, _bd)
        ) * inflation_B

        incremental_output_B = np.maximum(
            _sub_B["total_output_mwh"].to_numpy(dtype=float)
            - _sub_B["hydro_annual_mwh"].to_numpy(dtype=float),
            0.0,
        )

        variable_opex_B = (
            FINANCIAL.opex_balancing_eur_per_mwh
            * incremental_output_B
            * inflation_B
        )

        grid_tariffs_bess_charge_B = (
            (
                _sub_B["bess_charge_mwh"].to_numpy(dtype=float)
                - (
                    _sub_B["pv_annual_mwh"].to_numpy(dtype=float)
                    - _sub_B["pv_curtailed_mwh"].to_numpy(dtype=float)
                    + _sub_B["hydro_to_bess_mwh"].to_numpy(dtype=float)
                )
            )
            * FINANCIAL.grid_tariff_bess_charge_eur_per_mwh
        )

        revenue_B = (
            _sub_B["merchant_revenue_eur"].to_numpy(dtype=float)
            + _sub_B["ppa_delivered_mwh"].to_numpy(dtype=float) * ppa_price_eur_mwh
            - _sub_B["ppa_shortfall_mwh"].to_numpy(dtype=float) * da_prices_B
        )

        operating_cf_B, _, _, _ = compute_after_tax_operating_cf(
            revenue_eur=revenue_B,
            opex_eur=fixed_opex_B + variable_opex_B + grid_tariffs_bess_charge_B,
            depreciation_eur=depreciation_B,
        )

        # Scenario D
        years_D = _sub_D["year"].to_numpy(dtype=int)

        da_prices_D = (
            da_numeric[[str(y) for y in years_D]]
            .mean(axis=0)
            .to_numpy(dtype=float)
        )

        inflation_D = (1.0 + FINANCIAL.inflation_rate) ** (years_D - _PROJECT_START)

        fixed_opex_D = (
            FINANCIAL.opex_pv_eur_per_mw_year * _pv
            + get_bess_opex_eur_per_year(_bp, _bd)
        ) * inflation_D

        incremental_output_D = np.maximum(
            _sub_D["total_output_mwh"].to_numpy(dtype=float)
            - _sub_D["hydro_annual_mwh"].to_numpy(dtype=float),
            0.0,
        )

        variable_opex_D = (
            FINANCIAL.opex_balancing_eur_per_mwh
            * incremental_output_D
            * inflation_D
        )

        grid_tariffs_bess_charge_D = (
            (
                _sub_D["bess_charge_mwh"].to_numpy(dtype=float)
                - (
                    _sub_D["pv_annual_mwh"].to_numpy(dtype=float)
                    - _sub_D["pv_curtailed_mwh"].to_numpy(dtype=float)
                    + _sub_D["hydro_to_bess_mwh"].to_numpy(dtype=float)
                )
            )
            * FINANCIAL.grid_tariff_bess_charge_eur_per_mwh
        )

        hydro_idx_D = [base_years.index(y) for y in years_D]
        hydro_only_da_revenue_D = hydro_prod_all[hydro_idx_D] * da_prices_D

        revenue_D = (
            _sub_D["merchant_revenue_eur"].to_numpy(dtype=float)
            + _sub_D["ppa_delivered_mwh"].to_numpy(dtype=float) * ppa_price_eur_mwh
            - _sub_D["ppa_shortfall_mwh"].to_numpy(dtype=float) * da_prices_D
            - hydro_only_da_revenue_D
        )

        operating_cf_D, _, _, _ = compute_after_tax_operating_cf(
            revenue_eur=revenue_D,
            opex_eur=fixed_opex_D + variable_opex_D + grid_tariffs_bess_charge_D,
            depreciation_eur=depreciation_D,
        )

        project_cf = np.concatenate([
            cf_construction,
            operating_cf_B,
            operating_cf_D,
        ])

        return project_cf, total_capex_eur

    for _, _cfg in _configs.iterrows():
        _pv = float(_cfg["pv_mw"])
        _bp = float(_cfg["bess_power_mw"])
        _bd = float(_cfg["bess_duration_h"])
        _cs = float(_cfg["contract_share"])

        try:
            cf_low, total_capex_eur = build_project_cf_for_config(_cfg, ppa_min)
            irr_low = compute_project_irr(cf_low)

            cf_high, _ = build_project_cf_for_config(_cfg, ppa_max)
            irr_high = compute_project_irr(cf_high)

            if np.isnan(irr_low) or np.isnan(irr_high):
                required_ppa = np.nan
                status = "IRR calculation failed"

            elif irr_low >= target_irr:
                required_ppa = ppa_min
                status = "Already reaches target IRR"

            elif irr_high < target_irr:
                required_ppa = np.nan
                status = f"Target not reached even at {ppa_max:.1f} €/MWh"

            else:
                low = ppa_min
                high = ppa_max

                for _ in range(max_iter):
                    mid = (low + high) / 2.0

                    cf_mid, _ = build_project_cf_for_config(_cfg, mid)
                    irr_mid = compute_project_irr(cf_mid)

                    if abs(irr_mid - target_irr) < tol:
                        break

                    if irr_mid < target_irr:
                        low = mid
                    else:
                        high = mid

                required_ppa = mid
                status = "Solved"

            if np.isnan(required_ppa):
                final_irr = np.nan
                final_npv = np.nan
            else:
                final_cf, _ = build_project_cf_for_config(_cfg, required_ppa)
                final_irr = compute_project_irr(final_cf)
                final_npv = compute_npv(final_cf, target_irr)

        except Exception as e:
            required_ppa = np.nan
            final_irr = np.nan
            final_npv = np.nan
            total_capex_eur = np.nan
            status = f"Error: {e}"

        required_rows.append({
            "pv_mw": _pv,
            "bess_mw": _bp,
            "bess_h": _bd,
            "contract_share": _cs,
            "target_IRR": target_irr,
            "required_PPA_eur_MWh": required_ppa,
            "final_IRR": final_irr,
            f"NPV_at_target_IRR_kEUR": None if np.isnan(final_npv) else round(final_npv / 1e3),
            "capex_kEUR": None if np.isnan(total_capex_eur) else round(total_capex_eur / 1e3),
            "status": status,
        })

    df_required_ppa = (
        pd.DataFrame(required_rows)
        .sort_values("required_PPA_eur_MWh", ascending=True, na_position="last")
        .reset_index(drop=True)
    )

    df_required_ppa.index += 1

    return df_required_ppa

def compute_required_ppa_for_target_irr_range(
    target_irr_list=(0.07, 0.10, 0.12, 0.15),
    ppa_min: float = 0.0,
    ppa_max: float = 300.0,
    tol: float = 1e-5,
    max_iter: int = 100,
) -> pd.DataFrame:
    """
    Computes the required PPA price €/MWh for each configuration and
    for each target IRR in target_irr_list.

    Uses existing dispatch results. Dispatch is not re-optimized.
    """

    all_results = []

    for target_irr in target_irr_list:
        print(f"Computing required PPA for target IRR = {target_irr:.1%}")

        df_target = compute_required_ppa_for_target_irr(
            target_irr=target_irr,
            ppa_min=ppa_min,
            ppa_max=ppa_max,
            tol=tol,
            max_iter=max_iter,
        )

        df_target["target_IRR_pct"] = target_irr * 100

        all_results.append(df_target)

    df_required_ppa_range = (
        pd.concat(all_results, ignore_index=True)
        .sort_values(
            ["target_IRR", "required_PPA_eur_MWh"],
            ascending=[True, True],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    df_required_ppa_range.index += 1

    return df_required_ppa_range



df_required_ppa_range = compute_required_ppa_for_target_irr_range(
    target_irr_list=target_irr_range,
    ppa_min=0.0,
    ppa_max=300.0,
)

display(df_required_ppa_range)

df_required_ppa_range.to_csv(
    RESULTS_DIR / "required_PPA_for_IRR_range_optimal_scenario2.csv",
    index=True,
    encoding="utf-8-sig"
)
